In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

In [ ]:
TRAIN_CSV = "../data/alphabet/landmarks/train_landmarks_balanced.csv"
TEST_CSV = "../data/alphabet/landmarks/test_landmarks_merged.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train labels:")
display(train_df["label"].value_counts().sort_index())

print("Test labels:")
display(test_df["label"].value_counts().sort_index())

In [ ]:
def add_distance_features(df):
    result = df.copy()

    important_pairs = [
        (4, 8),
        (8, 12),
        (12, 16),
        (16, 20),

        (0, 4),
        (0, 8),
        (0, 12),
        (0, 16),
        (0, 20),

        (5, 8),
        (9, 12),
        (13, 16),
        (17, 20),

        (4, 20),
        (8, 20),
    ]

    for a, b in important_pairs:
        dx = result[f"x{a}"] - result[f"x{b}"]
        dy = result[f"y{a}"] - result[f"y{b}"]
        dz = result[f"z{a}"] - result[f"z{b}"]

        result[f"dist_{a}_{b}"] = np.sqrt(dx**2 + dy**2 + dz**2)

    return result

In [ ]:
DROP_COLUMNS = ["label", "file_path", "split"]

train_features_df = add_distance_features(train_df)
test_features_df = add_distance_features(test_df)

X_train = train_features_df.drop(columns=DROP_COLUMNS, errors="ignore")
y_train = train_features_df["label"]

X_test = test_features_df.drop(columns=DROP_COLUMNS, errors="ignore")
y_test = test_features_df["label"]

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Classes:", list(label_encoder.classes_))
print("Feature count:", X_train.shape[1])

In [ ]:
models = {
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="adam",
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=30,
            random_state=42
        ))
    ]),

    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            probability=True,
            random_state=42
        ))
    ]),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    )
}

In [ ]:
results = []

best_model = None
best_model_name = None
best_accuracy = 0

for name, clf in models.items():
    print(f"Training {name}...")

    clf.fit(X_train, y_train_encoded)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test_encoded, y_pred)

    results.append({
        "model": name,
        "accuracy": acc
    })

    print(f"{name} accuracy: {acc:.4f}")
    print()

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = clf
        best_model_name = name

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)

display(results_df)

print("Best model:", best_model_name)
print("Best accuracy:", best_accuracy)

In [ ]:
y_pred = best_model.predict(X_test)

print(f"Best model: {best_model_name}")
print(f"Test Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")

print(classification_report(
    y_test_encoded,
    y_pred,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test_encoded, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

fig, ax = plt.subplots(figsize=(12, 10))

disp.plot(
    cmap="Blues",
    xticks_rotation=45,
    values_format="d",
    ax=ax
)

plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../models/alphabet", exist_ok=True)

joblib.dump(best_model, "../models/alphabet/alphabet_best_landmark_model.pkl")
joblib.dump(label_encoder, "../models/alphabet/alphabet_label_encoder.pkl")

print("Saved:")
print("../models/alphabet/alphabet_best_landmark_model.pkl")
print("../models/alphabet/alphabet_label_encoder.pkl")